# 数学建模课堂练习：最优化问题

本 Notebook 使用 Python 3.10+ 完成两个建模任务：

1. 出租车数量估计及 Monte Carlo 模拟验证；
2. 飞机场塔台调度避碰优化。

运行结束后，关键结果表格与图片会保存到 `outputs` 文件夹，便于插入 Word 数学建模报告。

若环境缺少依赖，请在终端或 Notebook 单元格中执行：

```bash
pip install numpy pandas matplotlib scipy notebook ipykernel
```


# 0. 环境配置与依赖检查

In [ ]:
import sys
import importlib.util

print(f"Python 版本: {sys.version}")

required_packages = ["numpy", "pandas", "matplotlib", "scipy"]
missing_packages = [pkg for pkg in required_packages if importlib.util.find_spec(pkg) is None]

if missing_packages:
    print("缺少依赖:", missing_packages)
    print("请执行: pip install numpy pandas matplotlib scipy notebook ipykernel")
else:
    print("依赖检查通过: numpy, pandas, matplotlib, scipy 均已安装。")


In [ ]:
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
from scipy.optimize import minimize, differential_evolution

# 固定随机种子，保证模拟结果可复现
SEED = 20260514
rng = np.random.default_rng(SEED)

# 输出目录
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# matplotlib 中文字体自动检测；若无中文字体，则自动回退为英文标题，避免乱码
candidate_fonts = ["SimHei", "Microsoft YaHei", "Noto Sans CJK SC", "Source Han Sans SC", "Arial Unicode MS"]
installed_font_names = {font.name for font in font_manager.fontManager.ttflist}
available_chinese_fonts = [font for font in candidate_fonts if font in installed_font_names]

HAS_CHINESE_FONT = len(available_chinese_fonts) > 0
if HAS_CHINESE_FONT:
    plt.rcParams["font.sans-serif"] = [available_chinese_fonts[0]] + plt.rcParams.get("font.sans-serif", [])
    plt.rcParams["axes.unicode_minus"] = False
    print(f"已启用中文字体: {available_chinese_fonts[0]}")
else:
    print("未检测到常见中文字体，图像标题和坐标轴将使用英文。")

def text_label(chinese_text: str, english_text: str) -> str:
    """根据字体环境选择中文或英文图注。"""
    return chinese_text if HAS_CHINESE_FONT else english_text

plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 200


# 1. 出租车数量估计模型

出租车牌号从 `0101` 开始顺序发放。由于起始编号为 101，因此先将观测牌号转换为整数，再转换为相对编号：

$$
\text{relative\_id}=\text{plate\_number}-101+1
$$

使用最大编号修正估计模型：

$$
\hat{N}=\frac{n+1}{n}m-1
$$

其中 $n$ 为样本量，$m$ 为样本最大相对编号。

In [ ]:
plate_strings = ["0421", "0128", "0702", "0410", "0598", "0674", "0712", "0529", "0867", "0312"]
start_plate_number = 101

plate_numbers = np.array([int(plate) for plate in plate_strings], dtype=int)
relative_ids = plate_numbers - start_plate_number + 1

sample_size = len(relative_ids)
max_relative_id = int(relative_ids.max())
taxi_n_hat = ((sample_size + 1) / sample_size) * max_relative_id - 1
taxi_n_true_for_simulation = int(round(taxi_n_hat))

taxi_observations = pd.DataFrame({
    "plate_string": plate_strings,
    "plate_number": plate_numbers,
    "relative_id": relative_ids,
})

display(taxi_observations)
print(f"样本量 n = {sample_size}")
print(f"样本最大相对编号 m = {max_relative_id}")
print(f"出租车数量估计值 N_hat = {taxi_n_hat:.2f}，四舍五入后约为 {taxi_n_true_for_simulation} 辆")


# 2. 出租车估计的 Monte Carlo 模拟验证

假设真实出租车数量为 $N_{true}=\mathrm{round}(\hat{N})$。每次从 `1` 到 `N_true` 中不放回抽取 10 个编号，重复 10000 次，观察最大编号修正估计量的分布。

In [ ]:
def corrected_maximum_estimator(sample_max: float, n: int) -> float:
    """最大编号修正估计量。"""
    return ((n + 1) / n) * sample_max - 1

simulation_times = 10_000
taxi_estimates = np.empty(simulation_times)

for sim_idx in range(simulation_times):
    sampled_ids = rng.choice(np.arange(1, taxi_n_true_for_simulation + 1), size=sample_size, replace=False)
    taxi_estimates[sim_idx] = corrected_maximum_estimator(sampled_ids.max(), sample_size)

taxi_mc_summary = pd.DataFrame({
    "metric": ["mean", "std", "q05", "q95"],
    "value": [
        taxi_estimates.mean(),
        taxi_estimates.std(ddof=1),
        np.quantile(taxi_estimates, 0.05),
        np.quantile(taxi_estimates, 0.95),
    ],
})

display(taxi_mc_summary)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(taxi_estimates, bins=35, color="#4C78A8", edgecolor="white", alpha=0.85)
ax.axvline(taxi_n_true_for_simulation, color="#F58518", linestyle="--", linewidth=2, label=text_label("真实值 N_true", "True N"))
ax.axvline(taxi_n_hat, color="#E45756", linestyle="-", linewidth=2, label=text_label("原始样本估计值 N_hat", "Original estimate"))
ax.set_title(text_label("出租车数量估计的 Monte Carlo 分布", "Monte Carlo Distribution of Taxi Number Estimates"))
ax.set_xlabel(text_label("估计的出租车数量", "Estimated taxi count"))
ax.set_ylabel(text_label("频数", "Frequency"))
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "taxi_simulation_hist.png", bbox_inches="tight")
plt.show()


In [ ]:
sample_sizes = np.arange(5, 51)
sample_size_simulations = 3_000
mean_absolute_errors = []
root_mean_squared_errors = []

for n_current in sample_sizes:
    estimates_current = np.empty(sample_size_simulations)
    for sim_idx in range(sample_size_simulations):
        sampled_ids = rng.choice(np.arange(1, taxi_n_true_for_simulation + 1), size=n_current, replace=False)
        estimates_current[sim_idx] = corrected_maximum_estimator(sampled_ids.max(), n_current)
    errors = estimates_current - taxi_n_true_for_simulation
    mean_absolute_errors.append(np.mean(np.abs(errors)))
    root_mean_squared_errors.append(np.sqrt(np.mean(errors ** 2)))

taxi_sample_size_error = pd.DataFrame({
    "sample_size": sample_sizes,
    "mean_absolute_error": mean_absolute_errors,
    "root_mean_squared_error": root_mean_squared_errors,
})

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(sample_sizes, mean_absolute_errors, marker="o", linewidth=1.8, label=text_label("平均绝对误差", "MAE"))
ax.plot(sample_sizes, root_mean_squared_errors, marker="s", linewidth=1.8, label=text_label("均方根误差", "RMSE"))
ax.set_title(text_label("样本量对出租车数量估计误差的影响", "Effect of Sample Size on Estimation Error"))
ax.set_xlabel(text_label("样本量 n", "Sample size n"))
ax.set_ylabel(text_label("估计误差", "Estimation error"))
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "taxi_sample_size_error.png", bbox_inches="tight")
plt.show()

display(taxi_sample_size_error.head())


# 3. 飞机避碰优化模型

设第 $i$ 架飞机方向角调整量为 $\delta_i$，单位为度，约束为：

$$
-30 \leq \delta_i \leq 30
$$

调整后的方向角：

$$
\theta_i^{new}=\theta_i+\delta_i
$$

注意：输入方向角为“度”，计算三角函数前必须转换为“弧度”。速度向量为：

$$
v_i=800[\cos(\theta_i^{new}),\sin(\theta_i^{new})]
$$

目标函数为总调整量平方和最小：

$$
\min \sum_i \delta_i^2
$$

任意两机在时间区间 $[0,T]$ 内的最小距离必须不小于 8 km。

In [ ]:
aircraft_data = pd.DataFrame({
    "aircraft_id": [1, 2, 3, 4, 5, 6],
    "x_km": [150, 85, 150, 145, 130, 0],
    "y_km": [140, 85, 155, 50, 150, 0],
    "initial_angle_deg": [243, 236, 220.5, 159, 230, 52],
})

SPEED_KMH = 800.0
SAFE_DISTANCE_KM = 8.0
TIME_HORIZON_HOUR = 0.3
AIRSPACE_SIZE_KM = 160.0

positions = aircraft_data[["x_km", "y_km"]].to_numpy(dtype=float)
initial_angles_deg = aircraft_data["initial_angle_deg"].to_numpy(dtype=float)
aircraft_ids = aircraft_data["aircraft_id"].to_numpy(dtype=int)
num_aircraft = len(aircraft_data)
aircraft_pairs = list(combinations(range(num_aircraft), 2))

display(aircraft_data)


In [ ]:
def calculate_velocity_vectors(angle_degrees: np.ndarray, speed_kmh: float = SPEED_KMH) -> np.ndarray:
    """根据方向角计算速度向量。输入角度单位为度，三角函数计算前转换为弧度。"""
    angle_radians = np.deg2rad(angle_degrees)
    return speed_kmh * np.column_stack((np.cos(angle_radians), np.sin(angle_radians)))

def calculate_pair_min_distance(
    position_i: np.ndarray,
    position_j: np.ndarray,
    velocity_i: np.ndarray,
    velocity_j: np.ndarray,
    time_horizon: float = TIME_HORIZON_HOUR,
) -> tuple[float, float]:
    """计算两架飞机在 [0, T] 内的最小距离和对应时间。"""
    relative_position = position_i - position_j
    relative_velocity = velocity_i - velocity_j
    velocity_norm_squared = float(np.dot(relative_velocity, relative_velocity))

    if velocity_norm_squared < 1e-12:
        t_star = 0.0
    else:
        t_star = -float(np.dot(relative_position, relative_velocity)) / velocity_norm_squared
        t_star = float(np.clip(t_star, 0.0, time_horizon))

    closest_relative_position = relative_position + relative_velocity * t_star
    min_distance = float(np.linalg.norm(closest_relative_position))
    return min_distance, t_star

def calculate_pairwise_min_distance_matrix(
    positions_km: np.ndarray,
    velocity_vectors: np.ndarray,
    time_horizon: float = TIME_HORIZON_HOUR,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """计算所有飞机两两最小距离矩阵和对应时间矩阵。"""
    n_aircraft = positions_km.shape[0]
    distance_matrix = np.full((n_aircraft, n_aircraft), np.nan)
    time_matrix = np.full((n_aircraft, n_aircraft), np.nan)

    for i, j in combinations(range(n_aircraft), 2):
        distance, t_star = calculate_pair_min_distance(
            positions_km[i], positions_km[j], velocity_vectors[i], velocity_vectors[j], time_horizon
        )
        distance_matrix[i, j] = distance_matrix[j, i] = distance
        time_matrix[i, j] = time_matrix[j, i] = t_star

    labels = [f"A{aircraft_id}" for aircraft_id in aircraft_ids]
    distance_df = pd.DataFrame(distance_matrix, index=labels, columns=labels)
    time_df = pd.DataFrame(time_matrix, index=labels, columns=labels)
    return distance_df, time_df

def min_distance_summary(distance_df: pd.DataFrame) -> tuple[float, tuple[str, str]]:
    """返回矩阵中的最小非对角距离及对应飞机对。"""
    values = distance_df.to_numpy(dtype=float)
    min_flat_index = np.nanargmin(values)
    i, j = np.unravel_index(min_flat_index, values.shape)
    return float(values[i, j]), (distance_df.index[i], distance_df.columns[j])


# 4. 优化前碰撞风险分析

先令所有方向角调整量为 0，计算初始飞行方向下任意两机在 18 分钟内的最小距离。若存在最小距离小于 8 km 的飞机对，则存在碰撞风险。

In [ ]:
initial_velocity_vectors = calculate_velocity_vectors(initial_angles_deg)
min_distance_before_df, min_time_before_df = calculate_pairwise_min_distance_matrix(positions, initial_velocity_vectors)

min_distance_before, dangerous_pair_before = min_distance_summary(min_distance_before_df)
has_collision_risk_before = min_distance_before < SAFE_DISTANCE_KM

display(min_distance_before_df.round(3))
print(f"优化前最小两机距离: {min_distance_before:.3f} km，对应飞机对: {dangerous_pair_before}")
print(f"是否存在碰撞风险: {'是' if has_collision_risk_before else '否'}")


# 5. 方向角优化求解

使用 `scipy.optimize.minimize` 的 SLSQP 方法求解。约束条件为所有飞机对的最小距离均不小于 8 km。如果 SLSQP 未得到可行解，则先用 `differential_evolution` 进行全局搜索，再用 SLSQP 精修。

In [ ]:
def objective_function(delta_degrees: np.ndarray) -> float:
    """目标函数：最小化方向角调整量平方和。"""
    return float(np.sum(delta_degrees ** 2))

def velocities_from_delta(delta_degrees: np.ndarray) -> np.ndarray:
    """由调整量计算新的速度向量。"""
    adjusted_angles = initial_angles_deg + delta_degrees
    return calculate_velocity_vectors(adjusted_angles)

def pair_min_distance_from_delta(delta_degrees: np.ndarray, i: int, j: int) -> float:
    """给定调整量时，计算指定飞机对的最小距离。"""
    velocity_vectors = velocities_from_delta(delta_degrees)
    distance, _ = calculate_pair_min_distance(
        positions[i], positions[j], velocity_vectors[i], velocity_vectors[j], TIME_HORIZON_HOUR
    )
    return distance

def all_safety_margins(delta_degrees: np.ndarray) -> np.ndarray:
    """所有飞机对的安全距离裕度，非负表示满足约束。"""
    return np.array([
        pair_min_distance_from_delta(delta_degrees, i, j) - SAFE_DISTANCE_KM
        for i, j in aircraft_pairs
    ])

def is_feasible_solution(delta_degrees: np.ndarray, tolerance: float = 1e-5) -> bool:
    return bool(np.min(all_safety_margins(delta_degrees)) >= -tolerance)

bounds = [(-30.0, 30.0) for _ in range(num_aircraft)]
initial_delta = np.zeros(num_aircraft)

constraints = [
    {
        "type": "ineq",
        "fun": lambda delta, i=i, j=j: pair_min_distance_from_delta(delta, i, j) - SAFE_DISTANCE_KM,
    }
    for i, j in aircraft_pairs
]

slsqp_result = minimize(
    objective_function,
    x0=initial_delta,
    method="SLSQP",
    bounds=bounds,
    constraints=constraints,
    options={"ftol": 1e-9, "maxiter": 1000, "disp": False},
)

used_global_search = False
optimization_result = slsqp_result

if (not slsqp_result.success) or (not is_feasible_solution(slsqp_result.x)):
    used_global_search = True
    print("SLSQP 未得到可行解，开始 differential_evolution 全局搜索。")

    def penalty_objective(delta_degrees: np.ndarray) -> float:
        margins = all_safety_margins(delta_degrees)
        violations = np.minimum(margins, 0.0)
        return objective_function(delta_degrees) + 1e6 * float(np.sum(violations ** 2)) + 1e5 * float(np.sum(violations < 0))

    de_result = differential_evolution(
        penalty_objective,
        bounds=bounds,
        seed=SEED,
        tol=1e-7,
        maxiter=1200,
        polish=False,
        updating="immediate",
        workers=1,
    )

    refined_result = minimize(
        objective_function,
        x0=de_result.x,
        method="SLSQP",
        bounds=bounds,
        constraints=constraints,
        options={"ftol": 1e-9, "maxiter": 1000, "disp": False},
    )
    optimization_result = refined_result

optimal_delta_deg = optimization_result.x
adjusted_angles_deg = initial_angles_deg + optimal_delta_deg
optimized_velocity_vectors = calculate_velocity_vectors(adjusted_angles_deg)
min_distance_after_df, min_time_after_df = calculate_pairwise_min_distance_matrix(positions, optimized_velocity_vectors)
min_distance_after, dangerous_pair_after = min_distance_summary(min_distance_after_df)

optimization_success = bool(optimization_result.success and is_feasible_solution(optimal_delta_deg))

aircraft_optimization_results = aircraft_data.copy()
aircraft_optimization_results["optimal_delta_deg"] = optimal_delta_deg
aircraft_optimization_results["adjusted_angle_deg"] = adjusted_angles_deg

display(aircraft_optimization_results.round(4))
print(f"是否使用全局搜索: {'是' if used_global_search else '否'}")
print(f"优化是否成功且可行: {'是' if optimization_success else '否'}")
print(f"优化器消息: {optimization_result.message}")
print(f"目标函数值 sum(delta_i^2) = {objective_function(optimal_delta_deg):.6f}")
print(f"优化前最小两机距离 = {min_distance_before:.6f} km，飞机对: {dangerous_pair_before}")
print(f"优化后最小两机距离 = {min_distance_after:.6f} km，飞机对: {dangerous_pair_after}")


# 6. 优化结果可视化

In [ ]:
def closest_approach_points(i: int, j: int, velocity_vectors: np.ndarray) -> tuple[float, np.ndarray, np.ndarray, float]:
    """返回指定飞机对最近距离时刻、两机最近点坐标和最小距离。"""
    distance, t_star = calculate_pair_min_distance(
        positions[i], positions[j], velocity_vectors[i], velocity_vectors[j], TIME_HORIZON_HOUR
    )
    point_i = positions[i] + velocity_vectors[i] * t_star
    point_j = positions[j] + velocity_vectors[j] * t_star
    return t_star, point_i, point_j, distance


def highlight_critical_pairs(ax, velocity_vectors: np.ndarray, max_pairs: int = 3) -> None:
    """在轨迹图上标出安全裕度最小的飞机对、最近点和 8 km 安全圈。"""
    pair_records = []
    for i, j in aircraft_pairs:
        t_star, point_i, point_j, distance = closest_approach_points(i, j, velocity_vectors)
        pair_records.append((distance, t_star, i, j, point_i, point_j))

    pair_records.sort(key=lambda item: item[0])
    selected_pairs = [record for record in pair_records if record[0] <= SAFE_DISTANCE_KM + 0.10]
    if not selected_pairs:
        selected_pairs = pair_records[:1]
    else:
        selected_pairs = selected_pairs[:max_pairs]

    for rank, (distance, t_star, i, j, point_i, point_j) in enumerate(selected_pairs):
        color = "#D62728" if distance < SAFE_DISTANCE_KM - 1e-6 else "#2CA02C"
        ax.scatter([point_i[0], point_j[0]], [point_i[1], point_j[1]],
                   s=80, marker="x", color=color, linewidths=2.2, zorder=5)
        ax.plot([point_i[0], point_j[0]], [point_i[1], point_j[1]],
                color=color, linewidth=2.4, linestyle="-", alpha=0.95, zorder=4)
        safety_circle = plt.Circle(
            point_i, SAFE_DISTANCE_KM, fill=False, edgecolor=color,
            linestyle=":", linewidth=1.8, alpha=0.85, zorder=3
        )
        ax.add_patch(safety_circle)

        mid_point = (point_i + point_j) / 2
        ax.annotate(
            f"A{aircraft_ids[i]}-A{aircraft_ids[j]}\nd={distance:.2f} km\nt={t_star * 60:.1f} min",
            xy=mid_point,
            xytext=(mid_point[0] + 4 + 2 * rank, mid_point[1] + 4 + 2 * rank),
            fontsize=8,
            color=color,
            arrowprops=dict(arrowstyle="->", color=color, lw=1.0),
            bbox=dict(boxstyle="round,pad=0.25", fc="white", ec=color, alpha=0.88),
            zorder=6,
        )


def plot_aircraft_trajectories(
    angle_degrees: np.ndarray,
    title_cn: str,
    title_en: str,
    output_file: str,
    highlight_risk: bool = True,
) -> None:
    """绘制给定方向角下的飞机轨迹，并突出显示最近距离和安全圈。"""
    velocity_vectors = calculate_velocity_vectors(angle_degrees)
    time_grid = np.linspace(0.0, TIME_HORIZON_HOUR, 120)

    fig, ax = plt.subplots(figsize=(8.5, 7.2))
    ax.plot([0, AIRSPACE_SIZE_KM, AIRSPACE_SIZE_KM, 0, 0], [0, 0, AIRSPACE_SIZE_KM, AIRSPACE_SIZE_KM, 0],
            color="#555555", linewidth=1.2, linestyle="--", label=text_label("160 km 空域边界", "160 km airspace"))

    colors = plt.cm.tab10(np.linspace(0, 1, num_aircraft))
    all_points = []
    for idx, aircraft_id in enumerate(aircraft_ids):
        trajectory = positions[idx] + np.outer(time_grid, velocity_vectors[idx])
        all_points.append(trajectory)
        ax.plot(trajectory[:, 0], trajectory[:, 1], color=colors[idx], linewidth=2.1, label=f"A{aircraft_id}")
        ax.scatter(positions[idx, 0], positions[idx, 1], color=colors[idx], edgecolor="black", s=58, zorder=3)
        ax.scatter(trajectory[-1, 0], trajectory[-1, 1], color=colors[idx], marker="s", s=32, alpha=0.75, zorder=3)
        ax.text(positions[idx, 0] + 2, positions[idx, 1] + 2, f"A{aircraft_id}", fontsize=10, color=colors[idx], weight="bold")
        ax.annotate("", xy=trajectory[min(12, len(trajectory) - 1)], xytext=trajectory[0],
                    arrowprops=dict(arrowstyle="->", color=colors[idx], lw=1.5))

    if highlight_risk:
        highlight_critical_pairs(ax, velocity_vectors)

    all_points = np.vstack(all_points)
    min_x = min(0.0, float(all_points[:, 0].min())) - 12
    max_x = max(AIRSPACE_SIZE_KM, float(all_points[:, 0].max())) + 12
    min_y = min(0.0, float(all_points[:, 1].min())) - 12
    max_y = max(AIRSPACE_SIZE_KM, float(all_points[:, 1].max())) + 12
    ax.set_xlim(min_x, max_x)
    ax.set_ylim(min_y, max_y)
    ax.set_aspect("equal", adjustable="box")
    ax.set_title(text_label(title_cn, title_en))
    ax.set_xlabel(text_label("x 坐标 / km", "x / km"))
    ax.set_ylabel(text_label("y 坐标 / km", "y / km"))
    ax.legend(loc="best", fontsize=8)
    ax.grid(alpha=0.25)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / output_file, bbox_inches="tight")
    plt.show()

plot_aircraft_trajectories(
    initial_angles_deg,
    "优化前飞机轨迹图（标出危险最近点）",
    "Aircraft Trajectories Before Optimization with Closest Approach",
    "aircraft_trajectories_before.png",
)

plot_aircraft_trajectories(
    adjusted_angles_deg,
    "优化后飞机轨迹图（标出安全边界）",
    "Aircraft Trajectories After Optimization with Safety Boundary",
    "aircraft_trajectories_after.png",
)


In [ ]:
def plot_before_after_trajectory_comparison() -> None:
    """同一张图中叠加优化前后轨迹，突出有明显调整的飞机。"""
    time_grid = np.linspace(0.0, TIME_HORIZON_HOUR, 120)
    before_velocities = calculate_velocity_vectors(initial_angles_deg)
    after_velocities = calculate_velocity_vectors(adjusted_angles_deg)
    colors = plt.cm.tab10(np.linspace(0, 1, num_aircraft))

    fig, ax = plt.subplots(figsize=(8.5, 7.2))
    ax.plot([0, AIRSPACE_SIZE_KM, AIRSPACE_SIZE_KM, 0, 0], [0, 0, AIRSPACE_SIZE_KM, AIRSPACE_SIZE_KM, 0],
            color="#555555", linewidth=1.2, linestyle="--")

    all_points = []
    for idx, aircraft_id in enumerate(aircraft_ids):
        before_traj = positions[idx] + np.outer(time_grid, before_velocities[idx])
        after_traj = positions[idx] + np.outer(time_grid, after_velocities[idx])
        all_points.extend([before_traj, after_traj])
        changed = abs(optimal_delta_deg[idx]) > 0.05
        line_width = 2.7 if changed else 1.4
        alpha_before = 0.50 if changed else 0.25
        alpha_after = 0.95 if changed else 0.55
        ax.plot(before_traj[:, 0], before_traj[:, 1], color=colors[idx], linestyle="--",
                linewidth=line_width, alpha=alpha_before)
        ax.plot(after_traj[:, 0], after_traj[:, 1], color=colors[idx], linestyle="-",
                linewidth=line_width, alpha=alpha_after)
        ax.scatter(positions[idx, 0], positions[idx, 1], color=colors[idx], edgecolor="black", s=55, zorder=3)
        ax.text(positions[idx, 0] + 2, positions[idx, 1] + 2,
                f"A{aircraft_id}\nΔ={optimal_delta_deg[idx]:.2f}°", fontsize=8, color=colors[idx], weight="bold")

    highlight_critical_pairs(ax, before_velocities, max_pairs=1)
    highlight_critical_pairs(ax, after_velocities, max_pairs=1)

    all_points = np.vstack(all_points)
    ax.set_xlim(min(0.0, float(all_points[:, 0].min())) - 12, max(AIRSPACE_SIZE_KM, float(all_points[:, 0].max())) + 12)
    ax.set_ylim(min(0.0, float(all_points[:, 1].min())) - 12, max(AIRSPACE_SIZE_KM, float(all_points[:, 1].max())) + 12)
    ax.set_aspect("equal", adjustable="box")
    ax.set_title(text_label("优化前后轨迹叠加对比（虚线=优化前，实线=优化后）", "Before/After Trajectory Overlay"))
    ax.set_xlabel(text_label("x 坐标 / km", "x / km"))
    ax.set_ylabel(text_label("y 坐标 / km", "y / km"))
    ax.grid(alpha=0.25)

    from matplotlib.lines import Line2D
    legend_items = [
        Line2D([0], [0], color="black", linestyle="--", lw=2, label=text_label("优化前轨迹", "Before")),
        Line2D([0], [0], color="black", linestyle="-", lw=2, label=text_label("优化后轨迹", "After")),
        Line2D([0], [0], color="#D62728", linestyle="-", lw=2, label=text_label("不安全最近距离", "Unsafe closest distance")),
        Line2D([0], [0], color="#2CA02C", linestyle="-", lw=2, label=text_label("安全边界最近距离", "Safe boundary distance")),
    ]
    ax.legend(handles=legend_items, loc="best", fontsize=8)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "aircraft_trajectory_comparison.png", bbox_inches="tight")
    plt.show()


def plot_dangerous_pair_zoom() -> None:
    """围绕优化前最危险飞机对做局部放大，展示避碰前后的差异。"""
    before_velocities = calculate_velocity_vectors(initial_angles_deg)
    after_velocities = calculate_velocity_vectors(adjusted_angles_deg)
    pair_i = int(dangerous_pair_before[0].replace("A", "")) - 1
    pair_j = int(dangerous_pair_before[1].replace("A", "")) - 1
    states = [
        (text_label("优化前", "Before"), before_velocities, "#D62728"),
        (text_label("优化后", "After"), after_velocities, "#2CA02C"),
    ]

    fig, axes = plt.subplots(1, 2, figsize=(11, 5.2), sharex=False, sharey=False)
    for ax, (state_name, velocity_vectors, color) in zip(axes, states):
        t_star, point_i, point_j, distance = closest_approach_points(pair_i, pair_j, velocity_vectors)
        local_start = max(0.0, t_star - 0.035)
        local_end = min(TIME_HORIZON_HOUR, t_star + 0.035)
        if local_end - local_start < 0.03:
            local_start = max(0.0, t_star - 0.05)
            local_end = min(TIME_HORIZON_HOUR, t_star + 0.05)
        local_time_grid = np.linspace(local_start, local_end, 100)

        traj_i = positions[pair_i] + np.outer(local_time_grid, velocity_vectors[pair_i])
        traj_j = positions[pair_j] + np.outer(local_time_grid, velocity_vectors[pair_j])
        ax.plot(traj_i[:, 0], traj_i[:, 1], color="#4C78A8", linewidth=2.4, label=f"A{aircraft_ids[pair_i]}")
        ax.plot(traj_j[:, 0], traj_j[:, 1], color="#F58518", linewidth=2.4, label=f"A{aircraft_ids[pair_j]}")
        ax.scatter([point_i[0], point_j[0]], [point_i[1], point_j[1]], color=color, marker="x", s=95, linewidths=2.4, zorder=5)
        ax.plot([point_i[0], point_j[0]], [point_i[1], point_j[1]], color=color, linewidth=2.6, zorder=4)
        ax.add_patch(plt.Circle(point_i, SAFE_DISTANCE_KM, fill=False, edgecolor=color, linestyle=":", linewidth=2.0))
        ax.annotate(
            f"d_min={distance:.2f} km\nt={t_star * 60:.1f} min",
            xy=(point_i + point_j) / 2,
            xytext=((point_i[0] + point_j[0]) / 2 + 4, (point_i[1] + point_j[1]) / 2 + 4),
            fontsize=9,
            color=color,
            arrowprops=dict(arrowstyle="->", color=color),
            bbox=dict(boxstyle="round,pad=0.25", fc="white", ec=color, alpha=0.88),
        )

        local_points = np.vstack([traj_i, traj_j, point_i.reshape(1, 2), point_j.reshape(1, 2)])
        center = local_points.mean(axis=0)
        half_span = max(np.ptp(local_points[:, 0]), np.ptp(local_points[:, 1]), SAFE_DISTANCE_KM * 2.8) / 2 + 6
        ax.set_xlim(center[0] - half_span, center[0] + half_span)
        ax.set_ylim(center[1] - half_span, center[1] + half_span)
        ax.set_aspect("equal", adjustable="box")
        ax.set_title(f"{state_name}: A{aircraft_ids[pair_i]}-A{aircraft_ids[pair_j]}")
        ax.set_xlabel(text_label("x 坐标 / km", "x / km"))
        ax.set_ylabel(text_label("y 坐标 / km", "y / km"))
        ax.grid(alpha=0.25)
        ax.legend(loc="best", fontsize=8)

    fig.suptitle(text_label("最危险飞机对局部避碰效果放大图", "Zoomed Conflict Resolution for the Most Dangerous Pair"), y=1.02)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "dangerous_pair_conflict_zoom.png", bbox_inches="tight")
    plt.show()

plot_before_after_trajectory_comparison()
plot_dangerous_pair_zoom()


In [ ]:
def plot_min_distance_heatmap(distance_df: pd.DataFrame, title_cn: str, title_en: str, output_file: str) -> None:
    """使用 matplotlib 绘制两两最小距离热力图。"""
    matrix = distance_df.to_numpy(dtype=float)
    masked_matrix = np.ma.masked_invalid(matrix)

    fig, ax = plt.subplots(figsize=(7, 6))
    image = ax.imshow(masked_matrix, cmap="viridis")
    fig.colorbar(image, ax=ax, label=text_label("最小距离 / km", "Minimum distance / km"))
    ax.set_title(text_label(title_cn, title_en))
    ax.set_xticks(np.arange(len(distance_df.columns)))
    ax.set_yticks(np.arange(len(distance_df.index)))
    ax.set_xticklabels(distance_df.columns)
    ax.set_yticklabels(distance_df.index)

    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            if not np.isnan(matrix[i, j]):
                text_color = "white" if matrix[i, j] < np.nanmean(matrix) else "black"
                ax.text(j, i, f"{matrix[i, j]:.1f}", ha="center", va="center", color=text_color, fontsize=8)

    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / output_file, bbox_inches="tight")
    plt.show()

plot_min_distance_heatmap(
    min_distance_before_df,
    "优化前两两最小距离热力图",
    "Minimum Distance Heatmap Before Optimization",
    "min_distance_heatmap_before.png",
)

plot_min_distance_heatmap(
    min_distance_after_df,
    "优化后两两最小距离热力图",
    "Minimum Distance Heatmap After Optimization",
    "min_distance_heatmap_after.png",
)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
bar_colors = ["#4C78A8" if value >= 0 else "#E45756" for value in optimal_delta_deg]
ax.bar([f"A{aircraft_id}" for aircraft_id in aircraft_ids], optimal_delta_deg, color=bar_colors, edgecolor="black", linewidth=0.6)
ax.axhline(0, color="black", linewidth=1)
ax.axhline(30, color="#777777", linestyle="--", linewidth=1)
ax.axhline(-30, color="#777777", linestyle="--", linewidth=1)
ax.set_title(text_label("各飞机方向角最优调整量", "Optimal Angle Adjustments"))
ax.set_xlabel(text_label("飞机编号", "Aircraft"))
ax.set_ylabel(text_label("方向角调整量 / 度", "Angle adjustment / deg"))
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "angle_adjustment_bar.png", bbox_inches="tight")
plt.show()


In [ ]:
def distance_curve_for_pair(i: int, j: int, velocity_vectors: np.ndarray, time_grid: np.ndarray) -> np.ndarray:
    """计算指定飞机对在时间网格上的距离曲线。"""
    relative_position = positions[i] - positions[j]
    relative_velocity = velocity_vectors[i] - velocity_vectors[j]
    relative_positions_over_time = relative_position + np.outer(time_grid, relative_velocity)
    return np.linalg.norm(relative_positions_over_time, axis=1)

# 选择优化前最危险的飞机对，用于比较优化前后的距离随时间变化
dangerous_i = int(dangerous_pair_before[0].replace("A", "")) - 1
dangerous_j = int(dangerous_pair_before[1].replace("A", "")) - 1
time_grid = np.linspace(0.0, TIME_HORIZON_HOUR, 300)
time_minutes = time_grid * 60

distance_before_curve = distance_curve_for_pair(dangerous_i, dangerous_j, initial_velocity_vectors, time_grid)
distance_after_curve = distance_curve_for_pair(dangerous_i, dangerous_j, optimized_velocity_vectors, time_grid)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(time_minutes, distance_before_curve, linewidth=2, color="#E45756", label=text_label("优化前", "Before"))
ax.plot(time_minutes, distance_after_curve, linewidth=2, color="#4C78A8", label=text_label("优化后", "After"))
ax.axhline(SAFE_DISTANCE_KM, color="black", linestyle="--", linewidth=1.5, label=text_label("安全距离 8 km", "Safe distance 8 km"))
ax.set_title(text_label(
    f"最危险飞机对 {dangerous_pair_before[0]}-{dangerous_pair_before[1]} 的距离变化",
    f"Distance Curve of Most Dangerous Pair {dangerous_pair_before[0]}-{dangerous_pair_before[1]}",
))
ax.set_xlabel(text_label("时间 / 分钟", "Time / min"))
ax.set_ylabel(text_label("两机距离 / km", "Distance / km"))
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "dangerous_pair_distance_curve.png", bbox_inches="tight")
plt.show()


# 7. 结果导出

本节将关键结果表保存为 CSV 文件，并汇总已生成的图片文件路径。

In [ ]:
taxi_results = pd.DataFrame({
    "metric": [
        "sample_size",
        "max_relative_id",
        "taxi_n_hat",
        "taxi_n_true_for_simulation",
        "mc_mean",
        "mc_std",
        "mc_q05",
        "mc_q95",
    ],
    "value": [
        sample_size,
        max_relative_id,
        taxi_n_hat,
        taxi_n_true_for_simulation,
        taxi_estimates.mean(),
        taxi_estimates.std(ddof=1),
        np.quantile(taxi_estimates, 0.05),
        np.quantile(taxi_estimates, 0.95),
    ],
})

aircraft_optimization_export = aircraft_optimization_results.copy()
aircraft_optimization_export["optimization_success"] = optimization_success
aircraft_optimization_export["objective_value"] = objective_function(optimal_delta_deg)
aircraft_optimization_export["min_distance_before_km"] = min_distance_before
aircraft_optimization_export["min_distance_after_km"] = min_distance_after
aircraft_optimization_export["used_global_search"] = used_global_search

taxi_results.to_csv(OUTPUT_DIR / "taxi_results.csv", index=False, encoding="utf-8-sig")
aircraft_optimization_export.to_csv(OUTPUT_DIR / "aircraft_optimization_results.csv", index=False, encoding="utf-8-sig")
min_distance_before_df.to_csv(OUTPUT_DIR / "min_distance_before.csv", encoding="utf-8-sig")
min_distance_after_df.to_csv(OUTPUT_DIR / "min_distance_after.csv", encoding="utf-8-sig")

expected_output_files = [
    "taxi_results.csv",
    "aircraft_optimization_results.csv",
    "min_distance_before.csv",
    "min_distance_after.csv",
    "taxi_simulation_hist.png",
    "taxi_sample_size_error.png",
    "aircraft_trajectories_before.png",
    "aircraft_trajectories_after.png",
    "aircraft_trajectory_comparison.png",
    "dangerous_pair_conflict_zoom.png",
    "min_distance_heatmap_before.png",
    "min_distance_heatmap_after.png",
    "angle_adjustment_bar.png",
    "dangerous_pair_distance_curve.png",
]

output_summary = pd.DataFrame({
    "file": expected_output_files,
    "path": [str(OUTPUT_DIR / file) for file in expected_output_files],
    "exists_after_run": [(OUTPUT_DIR / file).exists() for file in expected_output_files],
})

display(output_summary)


# 8. 可写入建模报告的结论摘要

下面的代码会根据前面计算结果自动生成一段可写入数学建模报告的结论文字。

In [ ]:
taxi_mean = taxi_estimates.mean()
taxi_std = taxi_estimates.std(ddof=1)
taxi_q05 = np.quantile(taxi_estimates, 0.05)
taxi_q95 = np.quantile(taxi_estimates, 0.95)

summary_text = f"""
结论摘要：

1. 出租车数量估计：将观测牌号转换为相对编号后，样本最大相对编号为 {max_relative_id}，样本量为 {sample_size}。
   根据最大编号修正估计模型，估计该城市出租车数量为 {taxi_n_hat:.2f}，约为 {taxi_n_true_for_simulation} 辆。
   在 N_true={taxi_n_true_for_simulation} 的假设下进行 10000 次 Monte Carlo 模拟，估计值均值为 {taxi_mean:.2f}，
   标准差为 {taxi_std:.2f}，5% 分位数为 {taxi_q05:.2f}，95% 分位数为 {taxi_q95:.2f}。

2. 飞机避碰优化：优化前所有飞机对中的最小距离为 {min_distance_before:.3f} km，对应飞机对为 {dangerous_pair_before[0]}-{dangerous_pair_before[1]}。
   由于安全距离要求为 {SAFE_DISTANCE_KM:.1f} km，优化前{'存在' if has_collision_risk_before else '不存在'}碰撞风险。
   通过以方向角调整量平方和最小为目标的 SLSQP 优化，得到的目标函数值为 {objective_function(optimal_delta_deg):.6f}。
   优化后所有飞机对中的最小距离为 {min_distance_after:.3f} km，对应飞机对为 {dangerous_pair_after[0]}-{dangerous_pair_after[1]}。
   优化结果{'满足' if optimization_success else '未完全满足'}方向角调整范围和最小安全距离约束。
"""

print(summary_text)
